### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: pdf 1.pdf
  ✓ Loaded 15 pages

Processing: pdf 2.pdf
  ✓ Loaded 16 pages

Processing: pdf 3.pdf
  ✓ Loaded 9 pages

Processing: pdf 4.pdf
  ✓ Loaded 19 pages

Total documents loaded: 59


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 17.0; modified using iText® 5.5.13.4 ©2000-2024 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'Adobe InDesign 18.3 (Windows)', 'creationdate': '2025-09-22T18:00:46+02:00', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'moddate': '2025-10-17T16:24:15+02:00', 'trapped': '/False', 'doi': '10.1038/s41598-025-20224-1', 'source': '..\\data\\pdf\\pdf 1.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'pdf 1.pdf', 'file_type': 'pdf'}, page_content='Deep learning for text \nsummarization using NLP for \nautomated news digest\nK. M. Rani Krishna1, K. Somasundaram1\uf02a, P . Arulmozhivarman2\uf02a, Sarah A. Immanuel3 &  \nE. R. Rajkumar4\nText Summarization, a vital aspect of natural language processing, aims to condense text while \nretaining its essential meaning. This process is achieved throug

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 59 documents into 273 chunks

Example chunk:
Content: Deep learning for text 
summarization using NLP for 
automated news digest
K. M. Rani Krishna1, K. Somasundaram1, P . Arulmozhivarman2, Sarah A. Immanuel3 &  
E. R. Rajkumar4
Text Summarization, a v...
Metadata: {'producer': 'Adobe PDF Library 17.0; modified using iText® 5.5.13.4 ©2000-2024 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'Adobe InDesign 18.3 (Windows)', 'creationdate': '2025-09-22T18:00:46+02:00', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'moddate': '2025-10-17T16:24:15+02:00', 'trapped': '/False', 'doi': '10.1038/s41598-025-20224-1', 'source': '..\\data\\pdf\\pdf 1.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'pdf 1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 17.0; modified using iText® 5.5.13.4 ©2000-2024 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'Adobe InDesign 18.3 (Windows)', 'creationdate': '2025-09-22T18:00:46+02:00', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'moddate': '2025-10-17T16:24:15+02:00', 'trapped': '/False', 'doi': '10.1038/s41598-025-20224-1', 'source': '..\\data\\pdf\\pdf 1.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'pdf 1.pdf', 'file_type': 'pdf'}, page_content='Deep learning for text \nsummarization using NLP for \nautomated news digest\nK. M. Rani Krishna1, K. Somasundaram1\uf02a, P . Arulmozhivarman2\uf02a, Sarah A. Immanuel3 &  \nE. R. Rajkumar4\nText Summarization, a vital aspect of natural language processing, aims to condense text while \nretaining its essential meaning. This process is achieved throug

### embedding And vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1802.63it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\syedz\AppData\Local\Temp\ipykernel_23272\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 17.0; modified using iText® 5.5.13.4 ©2000-2024 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'Adobe InDesign 18.3 (Windows)', 'creationdate': '2025-09-22T18:00:46+02:00', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'moddate': '2025-10-17T16:24:15+02:00', 'trapped': '/False', 'doi': '10.1038/s41598-025-20224-1', 'source': '..\\data\\pdf\\pdf 1.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'pdf 1.pdf', 'file_type': 'pdf'}, page_content='Deep learning for text \nsummarization using NLP for \nautomated news digest\nK. M. Rani Krishna1, K. Somasundaram1\uf02a, P . Arulmozhivarman2\uf02a, Sarah A. Immanuel3 &  \nE. R. Rajkumar4\nText Summarization, a vital aspect of natural language processing, aims to condense text while \nretaining its essential meaning. This process is achieved throug

In [11]:
### convert the text to embeddings
texts =[doc.page_content for doc in chunks]
texts

['Deep learning for text \nsummarization using NLP for \nautomated news digest\nK. M. Rani Krishna1, K. Somasundaram1\uf02a, P . Arulmozhivarman2\uf02a, Sarah A. Immanuel3 &  \nE. R. Rajkumar4\nText Summarization, a vital aspect of natural language processing, aims to condense text while \nretaining its essential meaning. This process is achieved through extractive and abstractive methods. \nDeep Learning faces challenges in this domain, including semantic understanding, preservation of \nmeaning, efficient handling of long documents, and ensuring coherence and grammatical correctness. \nDespite these challenges, deep learning offers advantages such as time saving, facilitating information \nretrieval, scalability, and content personalization. Also, deep learning faces the risk of losing important \ndetails, subjectivity in information selection, difficulty in handling complex texts, and variability in \nsummary quality. Addressing these challenges remains an ongoing focus of research 

In [12]:
### convert the text to embeddings
texts =[doc.page_content for doc in chunks]

### generate embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store  in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 273 texts...


Batches: 100%|██████████| 9/9 [00:26<00:00,  2.90s/it]


Generated embeddings with shape: (273, 384)
Adding 273 documents to vector store...
Successfully added 273 documents to vector store
Total documents in collection: 273


### Retriever Pipeline From VectorStore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("What is Text Summarization?")

Retrieving documents for query: 'What is Text Summarization?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.01s/it]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d010ea0f_3',
  'content': 'Text summarization\nA natural language processing (NLP) technique called text summarizing seeks to summarize a text while \nkeeping its main ideas and important details. Making a condensed version of the original text that highlights the \nkey concepts, ideas, and information is the aim of this. The task of text summary is displayed in Fig.\xa01. The two \nprimary methods are below:\n (a) Extractive summarization Here, the summary is formed by choosing certain sentences or phrases straight \nfrom the source text. These selected sentences are typically the ones that contain the most important infor-\nmation, based on criteria such as relevance, importance, and frequency of occurrence.\n (b) Abstractive summarization This involves generating a summary that may contain new words or sentences \nthat are absent from the source material. This method seeks to comprehend the text’s content and reword it \nbriefly, potentially using natural language gener